In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp

# read xlsx
# path = 'problem_2/problem_2.xlsx'
# try:
#     df = pd.read_excel(path, sheet_name='Solution_Excel', header=None)
# except FileNotFoundError:
#     path = path.split('/')[1]
# df = pd.read_excel(path, sheet_name='Solution_Excel', header=None)

In [92]:
patterns = []
pattern = [
    [0, 1],
    [0, 1],
    [1, 1],
]

rotated_patterns = [pattern, np.rot90(pattern).tolist(), np.rot90(pattern, 2).tolist(), np.rot90(pattern, 3).tolist()]
for i in range(4): patterns.append(rotated_patterns)
patterns = patterns

In [118]:
board = np.array([
    [0, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 1, 0, 0],
])

In [119]:
model = gp.Model("doodleFit")
x = {}

placements = np.ndarray(shape=(len(patterns), 4, board.shape[0], board.shape[1]), dtype=object)
for i, pattern in enumerate(patterns):
    for j, rotated_pattern in enumerate(pattern):
        for row in range(board.shape[0] - len(rotated_pattern) + 1):
            for col in range(board.shape[1] - len(rotated_pattern[0]) + 1):
                if ~np.any((board[row:row + len(rotated_pattern[0]), col:col + len(rotated_pattern)] == 1) & (rotated_pattern == 1)):
                    placements[i][j][row][col] = model.addVar(vtype=gp.GRB.BINARY, name=f'pattern_{i}_{j}_{row}_{col}')
    
for i, pattern in enumerate(patterns):
    model.addConstr(gp.quicksum(placements[i][j][row][col]
                    for j in range(len(pattern))
                    for row in range(board.shape[0] - len(pattern[j]) + 1)
                    for col in range(board.shape[1] - len(pattern[j][0]) + 1)) <= 1, 
                    f'one_rotation_piece_{i}')

for i in range(board.shape[0]):
    for j in range(board.shape[1]):
        terms = []
        terms.append(board[i][j])
        for p in range(len(patterns)):
            for r in range(4):
                for ii in range(len(patterns[p][r])):
                    for jj in range(len(patterns[p][r][0])):
                        if patterns[p][r][ii][jj] == 1:
                            if i - ii >= 0 and j - jj >= 0 and placements[p][r][i - ii][j - jj] is not None:
                                terms.append(placements[p][r][i - ii][j - jj])
        model.addConstr(gp.quicksum(terms) <= 1, f'no_overlap_{i}_{j}')


# Set objective: maximize the count of placed cells
model.setObjective(gp.quicksum(placements[p][r][i][j] 
                               for p in range(len(patterns))
                               for r in range(4)
                               for i in range(board.shape[0])
                               for j in range(board.shape[1])
                               if placements[p][r][i][j] is not None), gp.GRB.MAXIMIZE)

model.update()
model.getVars()

[<gurobi.Var pattern_0_0_0_0>,
 <gurobi.Var pattern_0_0_0_1>,
 <gurobi.Var pattern_0_0_0_2>,
 <gurobi.Var pattern_0_0_1_0>,
 <gurobi.Var pattern_0_0_1_1>,
 <gurobi.Var pattern_0_0_1_2>,
 <gurobi.Var pattern_0_1_0_0>,
 <gurobi.Var pattern_0_1_0_1>,
 <gurobi.Var pattern_0_1_1_0>,
 <gurobi.Var pattern_0_1_1_1>,
 <gurobi.Var pattern_0_1_2_0>,
 <gurobi.Var pattern_0_1_2_1>,
 <gurobi.Var pattern_0_2_0_0>,
 <gurobi.Var pattern_0_2_0_1>,
 <gurobi.Var pattern_0_2_0_2>,
 <gurobi.Var pattern_0_2_1_0>,
 <gurobi.Var pattern_0_2_1_1>,
 <gurobi.Var pattern_0_2_1_2>,
 <gurobi.Var pattern_0_3_0_0>,
 <gurobi.Var pattern_0_3_0_1>,
 <gurobi.Var pattern_0_3_1_0>,
 <gurobi.Var pattern_0_3_1_1>,
 <gurobi.Var pattern_0_3_2_0>,
 <gurobi.Var pattern_0_3_2_1>,
 <gurobi.Var pattern_1_0_0_0>,
 <gurobi.Var pattern_1_0_0_1>,
 <gurobi.Var pattern_1_0_0_2>,
 <gurobi.Var pattern_1_0_1_0>,
 <gurobi.Var pattern_1_0_1_1>,
 <gurobi.Var pattern_1_0_1_2>,
 <gurobi.Var pattern_1_1_0_0>,
 <gurobi.Var pattern_1_1_0_1>,
 <gurobi

In [120]:
model.update()
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: 12th Gen Intel(R) Core(TM) i5-12500H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 20 rows, 96 columns and 480 nonzeros
Model fingerprint: 0x028f78ec
Variable types: 0 continuous, 96 integer (96 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Found heuristic solution: objective 2.0000000
Presolve removed 12 rows and 80 columns
Presolve time: 0.00s
Presolved: 8 rows, 16 columns, 48 nonzeros
Found heuristic solution: objective 2.0000000
Variable types: 0 continuous, 16 integer (16 binary)

Root relaxation: cutoff, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap

In [117]:
# print("\nThe optimal solutions:")
if model.status == gp.GRB.INFEASIBLE:
    print('The model is infeasible; computing IIS')
    model.computeIIS()
    for c in model.getConstrs():
        if c.IISConstr:
            print('%s' % c.constrName)
if model.status == gp.GRB.OPTIMAL:
    for var in model.getVars():
        print(f"{var.VarName}: {var.X}")

# print(f"The optimal number of courses to take is:{model.objVal}")
# list_of_variables_defined_above = [total_sold, total_quality, expected_total_quality, rev_a, rev_b, rev_c, total_rev, prod_a, prod_b, used_a, used_b, unused_a, unused_b, total_cost, total_profit]
# list_of_variables_defined_above = [unused_a, unused_b]
# for x in list_of_variables_defined_above:
#     print(f"{x}: {x.getValue()}")
# for constr in model.getConstrs():
#     print(f"Constraint: {constr.ConstrName}, Dual Value: {constr.Pi}")

pattern_0_0_0_0: 0.0
pattern_0_0_0_1: 0.0
pattern_0_0_0_2: 0.0
pattern_0_0_1_0: 0.0
pattern_0_0_1_1: 0.0
pattern_0_0_1_2: 0.0
pattern_0_1_0_0: 0.0
pattern_0_1_0_1: 0.0
pattern_0_1_1_0: 0.0
pattern_0_1_1_1: 0.0
pattern_0_1_2_0: 0.0
pattern_0_1_2_1: 0.0
pattern_0_2_0_0: 1.0
pattern_0_2_0_1: 0.0
pattern_0_2_0_2: 0.0
pattern_0_2_1_0: 0.0
pattern_0_2_1_1: 0.0
pattern_0_2_1_2: 0.0
pattern_0_3_0_0: 0.0
pattern_0_3_0_1: 0.0
pattern_0_3_1_0: 0.0
pattern_0_3_1_1: 0.0
pattern_0_3_2_0: 0.0
pattern_0_3_2_1: 0.0
pattern_1_0_0_0: 0.0
pattern_1_0_0_1: 0.0
pattern_1_0_0_2: 0.0
pattern_1_0_1_0: 0.0
pattern_1_0_1_1: 0.0
pattern_1_0_1_2: 0.0
pattern_1_1_0_0: 0.0
pattern_1_1_0_1: 0.0
pattern_1_1_1_0: 0.0
pattern_1_1_1_1: 0.0
pattern_1_1_2_0: 0.0
pattern_1_1_2_1: 0.0
pattern_1_2_0_0: 0.0
pattern_1_2_0_1: 0.0
pattern_1_2_0_2: 0.0
pattern_1_2_1_0: 0.0
pattern_1_2_1_1: 0.0
pattern_1_2_1_2: 1.0
pattern_1_3_0_0: 0.0
pattern_1_3_0_1: 0.0
pattern_1_3_1_0: 0.0
pattern_1_3_1_1: 0.0
pattern_1_3_2_0: 0.0
pattern_1_3_2

In [77]:
for c in model.getConstrs():
  # print what the constraint is evaluated to
  if c.Slack < 1e-6:
    print('Constraint %s is active at solution point' % (c.ConstrName))

Constraint one_rotation_piece_0 is active at solution point
Constraint one_rotation_piece_1 is active at solution point
Constraint no_overlap_0_0 is active at solution point
Constraint no_overlap_0_1 is active at solution point
Constraint no_overlap_0_2 is active at solution point
Constraint no_overlap_1_0 is active at solution point
Constraint no_overlap_1_2 is active at solution point
Constraint no_overlap_2_0 is active at solution point
Constraint no_overlap_2_1 is active at solution point
Constraint no_overlap_2_2 is active at solution point


### write to excel

In [11]:
write_df = df.copy()
write_df.iloc[24, 1] = sum(list([int(val.X) for val in x.values()]))
write_df.iloc[1:8, 1] = list([int(val.X) for val in x.values()])
prerequisites_fullfilled = {}
for course, prerequisite_courses in prerequisites.iterrows():
    prerequisites_fullfilled[course] = 1
    for i, prerequisite_course in enumerate(prerequisite_courses):
        if prerequisite_course != 0 and courses.values[i] != course:
            if x[courses.values[i]].X == 0:
                prerequisites_fullfilled[course] = 0
                break
write_df.iloc[1:8, 7] = list(prerequisites_fullfilled.values())
write_df.iloc[19:22, 2] = [r.getValue() for r in requirements_actually_fulfilled]
write_df.iloc[0:8, 5] = write_df.iloc[0:8, 7]
write_df.iloc[0:8, 6:8] = np.nan

from openpyxl.styles import Font
from openpyxl import load_workbook

with pd.ExcelWriter(path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    write_df.to_excel(writer, sheet_name='Solution_Gurobi', index=False, header=False)
book = load_workbook(path)
sheet = book['Solution_Gurobi']


from openpyxl.utils import get_column_letter

source_sheet = book['Solution_Excel']
for i, column in enumerate(source_sheet.columns, start=1):
    letter = get_column_letter(i)
    width = source_sheet.column_dimensions[letter].width
    sheet.column_dimensions[letter].width = width


bold_font = Font(bold=True)

sheet['A25'].font = bold_font
sheet['B25'].font = bold_font

book.save(path)